# Loan Approval Prediction - Exploratory Data Analysis

This notebook performs EDA and preprocessing on the Loan Approval Dataset.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

ModuleNotFoundError: No module named 'pandas'

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/loan_approval_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Basic Information

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
print("Missing Values:")
df.isnull().sum()

## 3. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts = df['loan_status'].value_counts()
colors = ['#e74c3c', '#2ecc71']

axes[0].bar(['Rejected (0)', 'Approved (1)'], target_counts.values, color=colors)
axes[0].set_title('Loan Status Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=['Rejected', 'Approved'], autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Loan Status Proportion')

plt.tight_layout()
plt.savefig('../report/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Numerical Features Distribution

In [ ]:
numerical_cols = ['age', 'years_employed', 'annual_income', 'credit_score', 
                  'credit_history_years', 'savings_assets', 'current_debt',
                  'loan_amount', 'interest_rate']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    sns.histplot(data=df, x=col, hue='loan_status', kde=True, ax=axes[idx], palette=colors)
    axes[idx].set_title(f'{col} Distribution')
    axes[idx].legend(['Rejected', 'Approved'])

plt.tight_layout()
plt.savefig('../report/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Categorical Features Analysis

In [ ]:
categorical_cols = ['occupation_status', 'product_type', 'loan_intent']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, col in enumerate(categorical_cols):
    approval_rate = df.groupby(col)['loan_status'].mean().sort_values(ascending=False)
    approval_rate.plot(kind='bar', ax=axes[idx], color='#3498db', edgecolor='black')
    axes[idx].set_title(f'Approval Rate by {col}')
    axes[idx].set_ylabel('Approval Rate')
    axes[idx].set_ylim(0, 1)
    axes[idx].axhline(y=0.55, color='red', linestyle='--', label='Overall Rate')
    for i, v in enumerate(approval_rate.values):
        axes[idx].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=9)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../report/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../report/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
target_corr = numeric_df.corr()['loan_status'].drop('loan_status').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors_corr = ['#2ecc71' if x > 0 else '#e74c3c' for x in target_corr.values]
target_corr.plot(kind='barh', color=colors_corr, edgecolor='black')
plt.title('Feature Correlation with Target (loan_status)')
plt.xlabel('Correlation Coefficient')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('../report/target_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Features Deep Dive

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(x='loan_status', y='credit_score', data=df, ax=axes[0, 0], palette=colors)
axes[0, 0].set_title('Credit Score by Loan Status')
axes[0, 0].set_xticklabels(['Rejected', 'Approved'])

sns.boxplot(x='loan_status', y='annual_income', data=df, ax=axes[0, 1], palette=colors)
axes[0, 1].set_title('Annual Income by Loan Status')
axes[0, 1].set_xticklabels(['Rejected', 'Approved'])

sns.boxplot(x='loan_status', y='debt_to_income_ratio', data=df, ax=axes[1, 0], palette=colors)
axes[1, 0].set_title('Debt-to-Income Ratio by Loan Status')
axes[1, 0].set_xticklabels(['Rejected', 'Approved'])

sns.boxplot(x='loan_status', y='delinquencies_last_2yrs', data=df, ax=axes[1, 1], palette=colors)
axes[1, 1].set_title('Delinquencies by Loan Status')
axes[1, 1].set_xticklabels(['Rejected', 'Approved'])

plt.tight_layout()
plt.savefig('../report/key_features_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Derived Features Verification

In [ ]:
df['calc_dti'] = df['current_debt'] / df['annual_income']
df['calc_lti'] = df['loan_amount'] / df['annual_income']

print("debt_to_income_ratio vs current_debt/annual_income:")
print(f"Match: {(abs(df['debt_to_income_ratio'] - df['calc_dti']) < 0.01).mean():.2%}")

print("\nloan_to_income_ratio vs loan_amount/annual_income:")
print(f"Match: {(abs(df['loan_to_income_ratio'] - df['calc_lti']) < 0.01).mean():.2%}")

df = df.drop(columns=['calc_dti', 'calc_lti'])

## 9. Data Preprocessing

In [ ]:
drop_columns = ['customer_id', 'current_debt']
df_processed = df.drop(columns=drop_columns)
print(f"Dropped columns: {drop_columns}")
print(f"Shape after dropping: {df_processed.shape}")

In [ ]:
categorical_cols = ['occupation_status', 'product_type', 'loan_intent']
df_encoded = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=False)
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nNew columns: {df_encoded.columns.tolist()}")

In [ ]:
X = df_encoded.drop(columns=['loan_status'])
y = df_encoded['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTarget distribution in train: {y_train.value_counts(normalize=True).round(3).to_dict()}")
print(f"Target distribution in test: {y_test.value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
numerical_features = ['age', 'years_employed', 'annual_income', 'credit_score',
                      'credit_history_years', 'savings_assets', 'defaults_on_file',
                      'delinquencies_last_2yrs', 'derogatory_marks', 'loan_amount',
                      'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio',
                      'payment_to_income_ratio']

scaler = StandardScaler()
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test[numerical_features] = scaler.transform(X_test[numerical_features])

print("Scaling complete!")
X_train.head()

## 10. Save Processed Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

joblib.dump(scaler, '../data/processed/scaler.pkl')
joblib.dump(numerical_features, '../data/processed/numerical_features.pkl')
joblib.dump(X_train.columns.tolist(), '../data/processed/feature_names.pkl')

print("Saved:")
print("  - X_train.csv, X_test.csv")
print("  - y_train.csv, y_test.csv")
print("  - scaler.pkl")
print("  - numerical_features.pkl")
print("  - feature_names.pkl")

## Summary

**Dataset:** 50,000 loan applications with 20 features

**Key Findings:**
- Target is fairly balanced (55% approved, 45% rejected)
- Credit score has strongest positive correlation (+0.50)
- Delinquencies and debt-to-income ratio have strongest negative correlation (~-0.32)
- Education loans have highest approval rate (67%)
- Debt consolidation loans have lowest approval rate (37%)

**Preprocessing:**
- Dropped: customer_id (identifier), current_debt (redundant with ratio)
- One-hot encoded: occupation_status, product_type, loan_intent
- Scaled numerical features using StandardScaler
- Final features: 26